### Stuff to use during development and debugging

#### Distance Between Points

Compute the distance between points.  Arguments are strings printed as the representation of a GeoPandas Point object, _e.g._ "POINT (23.646 69.875)"

In [1]:
import re
from math import sqrt

In [2]:
s = "POINT (23.646 69.875)"

In [3]:
re.findall(r'\d+\.\d+', s)

['23.646', '69.875']

In [4]:
[float(n) for n in re.findall(r'\d+\.\d+', s)]

[23.646, 69.875]

In [5]:
def gp_dist(p1, p2):
    x1, y1 = [float(n) for n in re.findall(r'\d+\.\d+', p1)]
    x2, y2 = [float(n) for n in re.findall(r'\d+\.\d+', p2)]
    return sqrt((x1-x2)**2 + (y1-y2)**2)

In [6]:
gp_dist('POINT (1.0 1.0)', 'POINT (0.0 0.0)')

1.4142135623730951

Why is `prg1_ep342` closer to IJ than NO?  Part of the answer: the line to NO is filtered out.  Correctly, it turns out.

What about OP?

In [7]:
# # the head and tail of NO
# head = 'POINT (41.883 26.756)'
# tail = 'POINT (36.718 37.509)'

# # midpoint of the perpendicular intersection
# mid = 'POINT (36.52084 37.91947)'

# the head and tail of OP
head = 'POINT (36.718 37.509)'
tail = 'POINT (23.646 69.875)'

# midpoint of the perpendicular intersection
mid = 'POINT (36.7583 37.40921)'

In [8]:
# length of segment
gp_dist(head,tail)

34.90609029954515

In [9]:
# from head to midpoint
gp_dist(head, mid)

0.10762032382407764

In [10]:
# from midpoint to tail
gp_dist(mid, tail)

35.013710623327256

In [11]:
gp_dist(head,tail) - (gp_dist(head, mid) + gp_dist(mid, tail))

-0.215240647606187

### Operations on Classes

In [12]:
class Position:

    sheet = 'Position'

    class Cell:
        id = 'ID'                     
        object_name = 'Surpass Object'
        category = 'Category'         
        x_coord = 'Position X'        
        y_coord = 'Position Y'       

    class Measurement:
        id = 'Name'           
        category = 'Category' 
        x_coord = 'Position X'
        y_coord = 'Position Y'


In [13]:
def as_dict(cls):
    return { attr: val for (attr,val) in cls.__dict__.items() if not attr.startswith('_') }


In [14]:
Position.sheet

'Position'

In [15]:
Position.Cell.id

'ID'

In [16]:
getattr(Position.Cell, 'id')

'ID'

In [17]:
setattr(Position.Cell, 'id', 'foo')

In [18]:
Position.Cell.id

'foo'

In [19]:
Position.Cell.__dict__

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 5,
              'id': 'foo',
              'object_name': 'Surpass Object',
              'category': 'Category',
              'x_coord': 'Position X',
              'y_coord': 'Position Y',
              '__static_attributes__': (),
              '__dict__': <attribute '__dict__' of 'Cell' objects>,
              '__weakref__': <attribute '__weakref__' of 'Cell' objects>,
              '__doc__': None})

In [20]:
{ attr: val for (attr,val) in Position.Cell.__dict__.items() if not attr.startswith('_') }

{'id': 'foo',
 'object_name': 'Surpass Object',
 'category': 'Category',
 'x_coord': 'Position X',
 'y_coord': 'Position Y'}

In [21]:
as_dict(Position.Cell)

{'id': 'foo',
 'object_name': 'Surpass Object',
 'category': 'Category',
 'x_coord': 'Position X',
 'y_coord': 'Position Y'}

In [22]:
as_dict(Position)

{'sheet': 'Position',
 'Cell': __main__.Position.Cell,
 'Measurement': __main__.Position.Measurement}

In [23]:
{'a','b','c'} | {'c','d','e'}

{'a', 'b', 'c', 'd', 'e'}

### Data Classes

A better way to define the configuration classes

In [24]:
from dataclasses import dataclass, asdict, fields, field

Define a data class for each section of a config file.  If a section `C2` is nested inside another section `C1` define `C2` as a member of `C1` (_e.g._ a `position` section has subsections for `cell` and `measurement`).

In [25]:
@dataclass
class Cell:
    id_col_1: str = 'Surpass Object'
    id_col_2: str = 'ID'                     
    x_coord: str = 'Position X'        
    y_coord: str = 'Position Y'       

@dataclass
class Measurement:
    name_col: str = 'Name'           
    x_coord: str = 'Position X'
    y_coord: str = 'Position Y'

@dataclass
class Position:
    sheet: str = 'Position'
    category_col: str = 'Category'
    cell_category: str = 'Surface'
    measurement_category: str = 'MeasurementPoint'
    cell: Cell = field(default_factory=Cell)
    measurement: Measurement = field(default_factory=Measurement)

@dataclass
class MeioticStage:
    id_col: str = 'GonadID'
    stage_names: list[str] = field(default_factory=lambda: ['TZ_start','TZ_end','Pachy_end'])

@dataclass
class Imaris:
    data: str = ''
    measurements: str = ''

@dataclass
class Output:
    data: str = ''
    log: str = ''

@dataclass
class Config:
    position: Position = field(default_factory=Position)
    meioticstage: MeioticStage = field(default_factory=MeioticStage)
    imaris: Imaris = field(default_factory=Imaris)
    output: Output = field(default_factory=Output)

    def update(self, other):
        Config._dfs(self, asdict(self), other)

    def _dfs(self, d1, d2):
        for k1, v1 in d1.items():
            if v2 := d2.get(k1):
                if isinstance(v1,dict) and isinstance(v2,dict):
                    cls = getattr(self, k1)
                    Config._dfs(cls, v1, v2)
                else:
                    setattr(self, k1, v2)


We need to be able to overwrite default values with values read from a TOML file.

The `replace` function in `dataclasses` does a shallow update, but we need a deep update for nested configuration sections.  To do that the top level Config class defines a deep update method.

To define the user's configuration settings:
- create an initial configuration with all the default settings by calling `Config()`
- load the user settings into a dictionary
- pass the dictionary to the config's `update` method

See examples in below.

Create a configuration with all the defaults:

In [26]:
cfg = Config()

In [27]:
fields(cfg)

(Field(name='position',type=<class '__main__.Position'>,default=<dataclasses._MISSING_TYPE object at 0x1064c1d30>,default_factory=<class '__main__.Position'>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=False,doc=None,_field_type=_FIELD),
 Field(name='meioticstage',type=<class '__main__.MeioticStage'>,default=<dataclasses._MISSING_TYPE object at 0x1064c1d30>,default_factory=<class '__main__.MeioticStage'>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=False,doc=None,_field_type=_FIELD),
 Field(name='imaris',type=<class '__main__.Imaris'>,default=<dataclasses._MISSING_TYPE object at 0x1064c1d30>,default_factory=<class '__main__.Imaris'>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=False,doc=None,_field_type=_FIELD),
 Field(name='output',type=<class '__main__.Output'>,default=<dataclasses._MISSING_TYPE object at 0x1064c1d30>,default_factory=<class '__main__.Output'>,init=True,repr=True,hash=None

In [28]:
asdict(cfg)

{'position': {'sheet': 'Position',
  'category_col': 'Category',
  'cell_category': 'Surface',
  'measurement_category': 'MeasurementPoint',
  'cell': {'id_col_1': 'Surpass Object',
   'id_col_2': 'ID',
   'x_coord': 'Position X',
   'y_coord': 'Position Y'},
  'measurement': {'name_col': 'Name',
   'x_coord': 'Position X',
   'y_coord': 'Position Y'}},
 'meioticstage': {'id_col': 'GonadID',
  'stage_names': ['TZ_start', 'TZ_end', 'Pachy_end']},
 'imaris': {'data': '', 'measurements': ''},
 'output': {'data': '', 'log': ''}}

In [29]:
cfg.position

Position(sheet='Position', category_col='Category', cell_category='Surface', measurement_category='MeasurementPoint', cell=Cell(id_col_1='Surpass Object', id_col_2='ID', x_coord='Position X', y_coord='Position Y'), measurement=Measurement(name_col='Name', x_coord='Position X', y_coord='Position Y'))

In [30]:
cfg.position.sheet

'Position'

In [31]:
cfg.position.cell.x_coord

'Position X'

In [32]:
cfg.imaris.data

''

### How to Load User Settings

In [33]:
from tomllib import loads

Get the settings from a TOML file:

In [34]:
spec = '''
[position]
sheet = 'Pos'

[position.cell]
x_coord = 'X'
y_coord = 'Y'

[position.measurement]
name_col = 'Z'

[meioticstage]
stage_names = ['A','B','C']

[imaris]
data = './data/PRG*'

[output]
data = '.'
'''

In [35]:
dct = loads(spec)

Pass the settings (now in a dict) to the `update` method:

In [36]:
cfg.update(dct)

The config object should have all of the original settings plus the new values for the settings in the user's config:

In [37]:
asdict(cfg)

{'position': {'sheet': 'Pos',
  'category_col': 'Category',
  'cell_category': 'Surface',
  'measurement_category': 'MeasurementPoint',
  'cell': {'id_col_1': 'Surpass Object',
   'id_col_2': 'ID',
   'x_coord': 'X',
   'y_coord': 'Y'},
  'measurement': {'name_col': 'Z',
   'x_coord': 'Position X',
   'y_coord': 'Position Y'}},
 'meioticstage': {'id_col': 'GonadID', 'stage_names': ['A', 'B', 'C']},
 'imaris': {'data': './data/PRG*', 'measurements': ''},
 'output': {'data': '.', 'log': ''}}

In [38]:
Cell

__main__.Cell

In [39]:
asdict(Cell())

{'id_col_1': 'Surpass Object',
 'id_col_2': 'ID',
 'x_coord': 'Position X',
 'y_coord': 'Position Y'}

In [40]:
asdict(Config())

{'position': {'sheet': 'Position',
  'category_col': 'Category',
  'cell_category': 'Surface',
  'measurement_category': 'MeasurementPoint',
  'cell': {'id_col_1': 'Surpass Object',
   'id_col_2': 'ID',
   'x_coord': 'Position X',
   'y_coord': 'Position Y'},
  'measurement': {'name_col': 'Name',
   'x_coord': 'Position X',
   'y_coord': 'Position Y'}},
 'meioticstage': {'id_col': 'GonadID',
  'stage_names': ['TZ_start', 'TZ_end', 'Pachy_end']},
 'imaris': {'data': '', 'measurements': ''},
 'output': {'data': '', 'log': ''}}

In [41]:
cfg.position

Position(sheet='Pos', category_col='Category', cell_category='Surface', measurement_category='MeasurementPoint', cell=Cell(id_col_1='Surpass Object', id_col_2='ID', x_coord='X', y_coord='Y'), measurement=Measurement(name_col='Z', x_coord='Position X', y_coord='Position Y'))

In [42]:
asdict(cfg.position)

{'sheet': 'Pos',
 'category_col': 'Category',
 'cell_category': 'Surface',
 'measurement_category': 'MeasurementPoint',
 'cell': {'id_col_1': 'Surpass Object',
  'id_col_2': 'ID',
  'x_coord': 'X',
  'y_coord': 'Y'},
 'measurement': {'name_col': 'Z',
  'x_coord': 'Position X',
  'y_coord': 'Position Y'}}